## Concept 1 — ChatMessageHistory + RunnableWithMessageHistory

### What is it?
The core modern memory pattern in LangChain 1.x built on two classes:

| Class | Role |
|---|---|
| `ChatMessageHistory` | An in-memory list of `HumanMessage` / `AIMessage` objects for one session |
| `RunnableWithMessageHistory` | Wraps any LCEL chain — auto-loads history before invoke, auto-saves after |

### How it works end-to-end
```
chain_with_history.invoke({"input": "..."}, config={"session_id": "..."})
  1. get_session_history(session_id)   ← load the right ChatMessageHistory from store
  2. inject messages → MessagesPlaceholder in prompt
  3. prompt | llm                      ← run the chain
  4. append HumanMessage + AIMessage back to store[session_id]
```

### Key concepts covered
- **Session isolation** — each `session_id` is a separate, independent conversation
- **Unbounded growth** — all history is injected on every call (no trimming)

### Limitation → what Concept 2 fixes
History grows forever. Long conversations eventually overflow the LLM context window.  
Concept 2 uses `trim_messages` to keep only the last K turns.

In [1]:
# Kernel alive-check — confirms the notebook kernel is running before any real code
print('hi')

hi


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')          # suppress noisy deprecation warnings
sys.path.insert(0, '.')                    # allow importing MemoryUtils.py from same folder

# Shared helpers across all Memory concepts
from MemoryUtils import get_llm, TEST_TURNS, TEST_QUERIES, run_recall_test
# get_llm()        → returns a configured LLM (Azure / Anthropic)
# TEST_TURNS       → list of 5 (human, ai) pairs used to pre-load conversation history
# TEST_QUERIES     → questions that probe recall of those 5 facts
# run_recall_test  → fires TEST_QUERIES, prints pass/fail per question

# Core memory classes
from langchain_community.chat_message_histories import ChatMessageHistory
# In-memory list of HumanMessage / AIMessage objects for one session

from langchain_core.runnables.history import RunnableWithMessageHistory
# Wraps any LCEL chain: auto-injects history before invoke, auto-saves after

# Prompt building blocks
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# MessagesPlaceholder → slot inside the prompt where history messages are injected

# Message types — used only for isinstance() checks when printing
from langchain_core.messages import HumanMessage, AIMessage

llm = get_llm()
print('LLM ready')

### Step 1 — `ChatMessageHistory`: the raw message store

`ChatMessageHistory` is a simple **list of messages** — no LLM, no chain, no persistence.  
It has two helper methods:
- `add_user_message(text)` → appends a `HumanMessage`
- `add_ai_message(text)` → appends an `AIMessage`

Access the raw list via `.messages`. This is what gets injected into the prompt later.

In [ ]:
history = ChatMessageHistory()           # empty message list for one session

# Manually add turns — simulates a short prior conversation
history.add_user_message('My name is Mohan.')       # appends HumanMessage
history.add_ai_message('Hello Mohan!')               # appends AIMessage
history.add_user_message('I work at Acme Corp.')
history.add_ai_message('Nice! What does Acme Corp do?')

print(f'Messages stored: {len(history.messages)}')   # should be 4

# Print each message with its role (Human or AI)
for msg in history.messages:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}] {msg.content}')

### Step 2 — `RunnableWithMessageHistory`: the auto-memory wrapper

This is the **glue layer** between your LCEL chain and the history store.

**What it does on every `.invoke()` call:**
1. Calls `get_session_history(session_id)` to fetch (or create) the right `ChatMessageHistory`
2. Fills the `MessagesPlaceholder('history')` slot in the prompt with those messages
3. Runs the chain: `prompt | llm`
4. Saves the new Human + AI turn back to the store automatically

**Three things to wire up:**
- `get_session_history` — a function returning a `ChatMessageHistory` for a given `session_id`
- `input_messages_key` — the key in `.invoke(dict)` that holds the current user question
- `history_messages_key` — the `variable_name` in `MessagesPlaceholder` inside the prompt

In [ ]:
# --- Storage layer: maps session_id (str) → ChatMessageHistory ---
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    # Called automatically by RunnableWithMessageHistory before every invoke
    # Creates a brand-new empty history if this session_id hasn't been seen before
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# --- Prompt: system context + history slot + current user question ---
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer using the conversation history.'),
    MessagesPlaceholder(variable_name='history'),   # ← all past messages injected here
    ('human', '{input}'),                            # ← current user turn
])

# Base LCEL chain (no memory yet — just prompt piped into LLM)
chain = prompt | llm

# --- Wrap the chain so history is managed automatically ---
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,            # lookup function: session_id → ChatMessageHistory
    input_messages_key='input',     # matches the '{input}' placeholder in the prompt
    history_messages_key='history', # matches variable_name in MessagesPlaceholder
)

print('Chain with history ready.')

### Step 3 — Build a conversation turn by turn

Each `.invoke()` call with the **same `session_id`** automatically sees all previous turns.  
After 5 turns (each = 1 Human + 1 AI message), the store holds **10 messages** for that session.

> `config = {'configurable': {'session_id': '...'}}` — the standard way to pass the session ID into `RunnableWithMessageHistory`.

In [ ]:
# session_id ties all calls together — same id = same memory bucket
config = {'configurable': {'session_id': 'mohan_demo'}}

# Feed all 5 TEST_TURNS through the chain; each call appends 2 messages to the store
for human, _ in TEST_TURNS:
    # RunnableWithMessageHistory auto-loads history, runs chain, auto-saves new turn
    chain_with_history.invoke({'input': human}, config)

# Inspect accumulated history for this session
session_history = get_session_history('mohan_demo')
print(f'Messages stored in session: {len(session_history.messages)}')  # 5 turns × 2 = 10

for msg in session_history.messages:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    # Truncate long AI responses for readability
    content = msg.content[:70] + '...' if len(msg.content) > 70 else msg.content
    print(f'  [{role}] {content}')

### Step 4 — Session isolation: different `session_id` = separate memory

Each `session_id` is an independent conversation with no shared state.  
This is how you'd support **multiple users** in one app — each user gets their own ID.

- `'mohan_demo'` → has 5 turns of history → LLM can answer personal questions  
- `'brand_new_session'` → empty history → LLM has no context and says so

In [ ]:
# 'mohan_demo' already has 5 turns → LLM knows the name from history
res_mohan = chain_with_history.invoke(
    {'input': 'What is my name?'},
    {'configurable': {'session_id': 'mohan_demo'}}
)

# 'brand_new_session' has zero history → LLM cannot answer
res_new = chain_with_history.invoke(
    {'input': 'What is my name?'},
    {'configurable': {'session_id': 'brand_new_session'}}   # new key → empty store entry
)

print('session_id=mohan_demo  :', res_mohan.content)   # expected: mentions "Mohan"
print('session_id=brand_new   :', res_new.content)      # expected: "I don't know" type answer

### Step 5 — `chat_with_history`: reusable helper for recall testing

Instead of generating AI responses via the LLM (which would be non-deterministic),  
we **directly inject** the pre-baked `TEST_TURNS` into the store.  
This gives a controlled, reproducible baseline for scoring memory recall.

In [ ]:
RECALL_SESSION = 'recall_test'

# Directly write TEST_TURNS into the store — bypasses the LLM for deterministic setup
# Each (human, ai) pair represents one full conversation turn
for human, ai in TEST_TURNS:
    get_session_history(RECALL_SESSION).add_user_message(human)  # known Human side
    get_session_history(RECALL_SESSION).add_ai_message(ai)       # known AI side

def chat_with_history(query: str) -> str:
    # Invokes the chain with the pre-loaded recall session
    # The full history (all 5 turns) is injected into the prompt on every call
    result = chain_with_history.invoke(
        {'input': query},
        {'configurable': {'session_id': RECALL_SESSION}}
    )
    return result.content

### Step 6 — Recall test: does the LLM remember all 5 facts?

`run_recall_test` fires each question from `TEST_QUERIES` and checks if the answer is in the LLM's response.

**Expected result here: perfect recall (5/5)**  
Because `RunnableWithMessageHistory` injects the **full unbounded history** into every prompt,  
all 5 facts from `TEST_TURNS` are always visible to the LLM.

> This is the trade-off: perfect recall now, but token cost grows linearly with conversation length.

In [ ]:
# Runs all TEST_QUERIES against the recall session and prints a pass/fail score
# chat_with_history is our query function; label appears in the printed output
run_recall_test(chat_with_history, label='Concept 1 -- ChatMessageHistory + RunnableWithMessageHistory')